## Part 4: Models, metrics and XAI (P4)

This part trains and compares the models on P3's modeling table, chooses the metrics, the hyperparameters and the decision threshold **without touching the test set**, evaluates the final choice once on the test set, explains the final model and lists its limitations.

**Inputs from Part 3:** the modeling table `model` (one row per driver per race, 1950-2024, with the `split` and `sample_weight` columns), the 12 final features `FEATURES` and their groups `FEATURE_GROUPS`.
**Outputs for P1's inference function:** `FINAL_MODEL` (preprocessing + model in one pipeline; it takes the columns in `FEATURES`), `FINAL_THRESHOLD` and `FINAL_NAME`.

Sections: 4.1) evaluation function, 4.2) data, weights, preprocessing and leakage checks, 4.3) sanity check of the metrics, 4.4) baselines, 4.5) statistical models (plus two other model families), 4.6) shallow neural network (plus a wider and a deeper network), 4.7) hyperparameter tuning (plus the number of trees and polynomial features), 4.8) metric choice and threshold, 4.9) final model, 4.10) test set evaluation, 4.11) explainability (XAI), 4.12) model limitations, 4.13) hand-off to the inference function.

Every attempt has its own cell and its own result, numbered **Attempt 1, 2, 3...** in the order they were run.

In [ ]:
# LOCAL SETUP (removed automatically when the notebooks are merged)
# In main.ipynb, Part 3 runs just before this part and defines `model`, FEATURES and FEATURE_GROUPS.
# Locally we get exactly the same objects by running P3's notebook first.
import sys
sys.path.append("../src")
from f1_clean import load_clean, SEED
from IPython.utils.capture import capture_output
with capture_output():          # hide Part 3's printouts and plots
    %run p3_features.ipynb
# Standalone fallback; the automatic merge drops this LOCAL SETUP cell,
# so the merged notebook uses P1's single authoritative flag.
if "EXCLUDE_SHARED_DRIVES" not in globals():
    EXCLUDE_SHARED_DRIVES = False


In [ ]:
clean = load_clean(exclude_shared_drives=EXCLUDE_SHARED_DRIVES)
outcomes = clean["outcomes"]    # post-race columns (status, finishing position...). Never features: only used to tell the stories in 4.11-4.12
tables = clean["tables"]        # cleaned tables (driver and race names for the examples)

### 4.1 The evaluation function `train_eval()`

One function scores every model the same way, so all results (ours and P3's ablation) are directly comparable.

**Why these metrics?** The brief asks for ROC-AUC, PR-AUC and F1 for every model; we also report precision and recall, because F1 alone hides which of the two is weak.
* **ROC-AUC:** how well the model separates podium from non-podium overall (uses probabilities, no threshold).
* **PR-AUC:** how well the model ranks the real podium finishers at the top (uses probabilities). Computed as *average precision*. It is more informative than ROC-AUC when the positive class is rare, and podiums are rare (about 14-15% of rows).
* **F1, precision, recall:** quality of the yes/no decision at a probability threshold. The threshold is 0.5 until it is tuned on validation in 4.8.

**Training weights (new):** P3 keeps every season from 1950 and gives the rows before 2006 a weight of 0.25 (`sample_weight`). `train_eval()` takes these weights as an optional argument: it passes them to the model's `fit` (for a pipeline, to its last step) and uses them for the training scores too, so the training score measures what the model was asked to learn. Validation and test rows all have weight 1, so their scores are plain. Without the argument the function works exactly as before, so P3's code that calls it does not change.

**Leakage safety:** the model is fitted on the training years only. The test set is deliberately *not* an input: it is scored once, at the very end, with `evaluate()`.

The same code is in `src/f1_eval.py`, so P3 can import it.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


def evaluate(model, X, y, threshold=0.5, sample_weight=None): 
  # It takes a model that’s already trained, gives it one dataset, and returns the 5 scores.
    """Score an already-fitted model on one split and return the 5 metrics.
    ROC-AUC and PR-AUC use the probabilities (no threshold needed);
    F1, precision and recall use the yes/no prediction at `threshold`.
    `sample_weight` (optional) weights the rows, as in training."""
    proba = model.predict_proba(X)[:, 1]          # predicted probability of a podium
    pred = (proba >= threshold).astype(int)       # 1 = "podium" at this threshold
    w = sample_weight
    return {
        "ROC-AUC": roc_auc_score(y, proba, sample_weight=w),
        "PR-AUC": average_precision_score(y, proba, sample_weight=w),
        "F1": f1_score(y, pred, sample_weight=w, zero_division=0),
        "precision": precision_score(y, pred, sample_weight=w, zero_division=0),
        "recall": recall_score(y, pred, sample_weight=w, zero_division=0),
    }


def train_eval(model, X_train, y_train, X_val, y_val, threshold=0.5, name="model", sample_weight=None):
    # Trains the model on the training years. Calls evaluate twice: once on train, once on validation.
    """Fit `model` on the training years only, then score it on train and validation.
    `sample_weight` (optional): one weight per training row (P3's `sample_weight` column). It is passed to fit
    (to the last step of a pipeline) and used for the training scores. Validation rows are not weighted.
    The test set is not used here on purpose: it is scored once, at the end, with evaluate()."""
    if sample_weight is None:
        model.fit(X_train, y_train)
    else:
        sample_weight = np.asarray(sample_weight, dtype=float)
        key = f"{model.steps[-1][0]}__sample_weight" if hasattr(model, "steps") else "sample_weight"
        model.fit(X_train, y_train, **{key: sample_weight})
    rows = [
        {"model": name, "split": "train", **evaluate(model, X_train, y_train, threshold, sample_weight)},
        {"model": name, "split": "validation", **evaluate(model, X_val, y_val, threshold)},
    ]
    return pd.DataFrame(rows).round(4)

### 4.2 Data, weights, preprocessing and leakage checks

**Data:** P3's modeling table and P3's time split: train 1950-2019, validation 2020-2021, test 2022-2024. The test rows are separated only in 4.10.

**Features:** P3's final `FEATURES`, 12 numeric columns in 4 groups (grid, qualifying, driver standings, constructor standings), chosen by P3's ablation (3.11). Everything P3 tested and dropped (circuit history, home race, era, recent form, raw lap time) is not used. In the previous version we had to remove `era_ordinal` ourselves; P3's final list no longer contains it.

**Weights:** every training row from 2006 on has weight 1, every row from 1950-2005 has weight 0.25 (P3, 3.3-3.4 and 3.11 Part F). Old seasons have little or no qualifying data and a different sport (more cars, more retirements, other points systems), so they count a quarter. Every model below gets the weights, except k-NN, which cannot use them (4.5).

**Preprocessing (one pipeline, the same for every model):**
* **Column selection:** the first step keeps the columns in `FEATURES` and turns them into plain numbers (P3 stores some as nullable integers). P1's inference function passes exactly these columns.
* **Missing values are structural, not errors:** the championship position is empty in round 1 of every season, and qualifying data does not exist before 1994 and is patchy until 2006. We fill gaps with the **training median** and add a **missing-indicator column** for every feature that has gaps in training, so the model still knows that the value was missing.
* **Scaling:** every feature is standardized (mean 0, standard deviation 1). Logistic regression, k-NN and the network need it; for the tree models it changes nothing.
* **Fitted on the training rows only:** the imputer and the scaler are steps of a `Pipeline`, so `fit()` learns the medians, means and standard deviations from the training years only; validation and test rows are only transformed. (The weights go to the model step; the imputer and scaler use every training row equally.)
* **Difference from P3's experiments:** P3's ablation uses plain median imputation without missing-value flags. Our models add the flags (`add_indicator=True`), so P3's logistic-regression numbers and ours differ slightly. P3 also scores on three validation periods; the team rule for P4 is the single validation period 2020-2021, so our numbers are not directly comparable with her averages.

**Leakage checks (code below):** no post-race column is a feature, the weight is not a feature, each split holds only its own years, and only training rows before 2006 have a weight below 1.

In [ ]:
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

table = model                     # P3's modeling table (read only; the name `model` is kept for Part 3)

# leakage checks: no post-race column and no weight among the features, and the time split is respected
assert not set(FEATURES) & set(outcomes.columns), "a post-race column is used as a feature"
assert "sample_weight" not in FEATURES, "the training weight must not be a feature"
assert table.loc[table["split"] == "train", "year"].max() <= 2019
assert table.loc[table["split"] == "validation", "year"].between(2020, 2021).all()
assert table.loc[table["split"] == "test", "year"].min() >= 2022
assert (table.loc[table["sample_weight"] < 1, "year"] < 2006).all(), "only pre-2006 rows may be down-weighted"

train = table[table["split"] == "train"]
val = table[table["split"] == "validation"]
X_train, y_train, w_train = train[FEATURES].astype(float), train["podium"], train["sample_weight"].to_numpy()
X_val, y_val = val[FEATURES].astype(float), val["podium"]


def keep_features(X):
    """First step of every model: keep the FEATURES columns as plain float numbers."""
    return X[FEATURES].astype(float)

select = FunctionTransformer(keep_features, feature_names_out=lambda transformer, names: FEATURES)
preprocess = make_pipeline(select, SimpleImputer(strategy="median", add_indicator=True), StandardScaler())


def with_preprocessing(estimator):
    """The shared preprocessing followed by `estimator`, as one pipeline (fitted on the training rows only)."""
    return Pipeline([("prep", clone(preprocess)), ("model", estimator)])


print(f"{len(FEATURES)} features in {len(FEATURE_GROUPS)} groups: {FEATURE_GROUPS}")
summary = table[table["split"] != "test"].groupby(["split", table["year"] >= 2006]).agg(
    rows=("podium", "size"), years=("year", lambda s: f"{s.min()}-{s.max()}"),
    podium_rate=("podium", "mean"), weight=("sample_weight", "mean"))
summary.index = summary.index.set_levels(["1950-2005", "2006 and later"], level=1)
display(summary.round(3))
missing = pd.DataFrame({"1950-2005": X_train[train["year"] < 2006].isna().mean(),
                        "2006-2019": X_train[train["year"] >= 2006].isna().mean()})
display(missing.loc[missing.max(axis=1) > 0].round(3).rename_axis("share missing (train)"))

**What we see.**
* **Rows and weights:** 22,441 training rows: 16,699 from 1950-2005 (weight 0.25, podium rate 13.4%) and 5,742 from 2006-2019 (weight 1, 14.0%). Validation: 774 rows from 2020-2021 (15.1%). In weighted terms the old seasons count as about 4,175 rows, a bit less than the modern ones.
* **Why the old seasons get a lower weight:** about 85% of the 1950-2005 rows have no qualifying position or gap (qualifying data starts in 1994 and is patchy until 2006), against less than 2% from 2006 on. The championship position is empty more often too (14% vs 7%), because the early seasons were short.
* **The missing-value flags matter now:** for the old rows, "qualifying missing" is the normal case, not a rare signal. The model sees it through the flags of 4.2.

### 4.3 Sanity check of the metrics

Before trusting the function, we test it on a model that knows nothing: `DummyClassifier` always predicts the training podium rate. A correct implementation must give **ROC-AUC = 0.5** (no separation), **PR-AUC = the podium rate** (no better than random ranking) and **F1 = 0** at threshold 0.5 (it never says "podium"). We also print its accuracy, to show why accuracy is not one of our metrics.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
# dummy da msh 3arf ay haga gher average msh arf ay haga 3an features 
# bnshof what evaluation metrics we will choose to evaluate the model on
dummy = DummyClassifier(strategy="prior", random_state=SEED)
dummy_results = train_eval(dummy, X_train, y_train, X_val, y_val, name="dummy (always the base rate)",
                           sample_weight=w_train)
display(dummy_results)

val_row = dummy_results[dummy_results["split"] == "validation"].iloc[0]
assert abs(val_row["ROC-AUC"] - 0.5) < 1e-3
assert abs(val_row["PR-AUC"] - y_val.mean()) < 1e-3
assert val_row["F1"] == 0
print(f"Dummy accuracy on validation: {accuracy_score(y_val, dummy.predict(X_val)):.1%} "
      f"(looks good, but it finds 0 podiums)")

#usually accuracy is not a good metric for imbalanced datasets, so we will use F1, PR-AUC and ROC-AUC to evaluate our models

**What we see.** The dummy model gets ROC-AUC 0.5, PR-AUC equal to the podium rate (0.151 on validation) and F1 = 0, exactly as expected, so the function computes the metrics correctly (with and without weights). Its accuracy is still 84.9%, although it never finds a single podium. That is why accuracy is not one of our metrics, and why **PR-AUC is the natural primary metric** for this rare positive class. These numbers are the floor every real model must beat.

### 4.4 Baselines

A baseline is the simplest reasonable approach. Every later model must beat it, otherwise its extra complexity is not earned.
* **Rule `grid <= 3`:** predict a podium for the three cars that start at the front. It learns nothing; it encodes the domain fact that the starting position is the strongest single predictor.
* **Logistic regression:** the simplest statistical model. It multiplies each feature by a learned weight, adds them up, and turns the sum into a probability with the sigmoid function. *Limitation:* it is linear, so it cannot learn interactions on its own (for example "a good grid slot matters more for a top team").

#### Attempt 1: rule `grid <= 3`
The rule gives a hard yes/no, so its ROC-AUC and PR-AUC are those of a single operating point. It uses P3's `grid_fixed`, where a pit-lane start is slot 35, so a pit-lane start is never a front-row start.

In [ ]:
class GridRule:
    """Rule baseline: 'podium' if the car starts in grid slot 1, 2 or 3. Learns nothing."""
    def fit(self, X, y, sample_weight=None):
        return self

    def predict_proba(self, X):
        p = X["grid_fixed"].between(1, 3).astype(float).to_numpy()
        return np.column_stack([1 - p, p])

rule_results = train_eval(GridRule(), X_train, y_train, X_val, y_val, name="rule grid <= 3", sample_weight=w_train)
display(rule_results)

#### Attempt 2: logistic regression (12 features, with the weights)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = with_preprocessing(LogisticRegression(max_iter=2000, random_state=SEED))
lr_results = train_eval(lr, X_train, y_train, X_val, y_val, name="logistic regression", sample_weight=w_train)
display(lr_results)

#### Attempt 3: the same logistic regression without the weights
Every row from 1950 counts fully. Does P3's down-weighting of the old seasons also help our pipeline?

In [ ]:
lr_noweight = with_preprocessing(LogisticRegression(max_iter=2000, random_state=SEED))
lr_noweight_results = train_eval(lr_noweight, X_train, y_train, X_val, y_val,
                                 name="logistic regression (no weights)")
display(lr_noweight_results)

#### Attempt 4: the same logistic regression on 2006-2019 only
The opposite extreme: old seasons get weight 0 (the setup of our previous version).

In [ ]:
modern = (train["year"] >= 2006).to_numpy()
lr_modern = with_preprocessing(LogisticRegression(max_iter=2000, random_state=SEED))
lr_modern_results = train_eval(lr_modern, X_train[modern], y_train[modern], X_val, y_val,
                               name="logistic regression (2006-2019 only)")
display(lr_modern_results)

**What we see (the weights, Attempts 2-4).**
* **Weighted (P3's setup): 0.736.** All rows at full weight: **0.728**. Only 2006-2019: **0.744**.
* **The weights help** compared with treating 1950s races like modern ones (+0.008).
* **Dropping the old seasons scores a little higher still** (+0.008 over the weighted version). P3 found the same on her three validation periods and documented it: the team keeps the old seasons with a low weight because the course asks us not to simply drop them.
* **All three numbers are within 0.016 of each other,** far less than the noise of the validation PR-AUC (standard error about 0.037, 4.9). We follow P3's setup (weighted) for every model below.

#### Attempt 5: logistic regression with `class_weight="balanced"`
Podiums are only about 14% of the training rows. `class_weight="balanced"` gives the podium rows a larger weight, so both classes count equally in training (it multiplies with P3's season weights). We expect the ranking metrics (ROC-AUC, PR-AUC) to stay about the same and the yes/no metrics at threshold 0.5 to change, because the weighting mostly moves the predicted probabilities up.

In [ ]:
lr_bal = with_preprocessing(LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
lr_bal_results = train_eval(lr_bal, X_train, y_train, X_val, y_val, name="logistic regression (balanced)",
                            sample_weight=w_train)
display(lr_bal_results)

#### Attempt 6: logistic regression on the 4 draft features only
Before P3's table was ready, the models were built on four raw pre-race columns (`grid`, `pitlane_start`, `has_qualifying`, `qual_position`). Same model, same rows, same weights: does P3's feature engineering add anything?

In [ ]:
DRAFT_FEATURES = ["grid", "pitlane_start", "has_qualifying", "qual_position"]
lr_draft = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(max_iter=2000, random_state=SEED))
lr_draft_results = train_eval(lr_draft, train[DRAFT_FEATURES].astype(float), y_train,
                              val[DRAFT_FEATURES].astype(float), y_val,
                              name="logistic regression (4 draft features)", sample_weight=w_train)
display(lr_draft_results)

#### Comparison of the baselines (validation)

In [ ]:
import matplotlib.pyplot as plt

baselines = pd.concat([dummy_results, rule_results, lr_results, lr_noweight_results, lr_modern_results,
                       lr_bal_results, lr_draft_results], ignore_index=True)
val_table = baselines[baselines["split"] == "validation"].set_index("model").drop(columns="split")
display(val_table)

ax = val_table[["ROC-AUC", "PR-AUC", "F1"]].plot.barh(figsize=(8, 4.5))
ax.set_title("Baselines on validation (2020-2021), threshold 0.5")
ax.set_xlabel("score (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(title="metric", loc="lower right")
plt.tight_layout()
plt.show()

# what the logistic regression learned: weights of the standardized inputs, largest first
weights = pd.Series(lr.named_steps["model"].coef_[0], index=lr.named_steps["prep"].get_feature_names_out())
display(weights.sort_values(key=abs, ascending=False).head(10).round(3).to_frame("weight (standardized input)"))

**What we see (validation 2020-2021).**
* **Logistic regression is far above both references:** PR-AUC 0.736 vs 0.530 for the rule and 0.151 for the dummy; ROC-AUC 0.929 vs 0.820.
* **P3's features add real information:** the same weighted logistic regression reaches PR-AUC 0.667 with the 4 draft features and **0.736 with P3's 12 features (+0.069)**.
* **At threshold 0.5 the rule still has the best F1** (0.695 vs 0.648): plain logistic regression is cautious at 0.5 (precision 0.73, recall 0.58). The threshold is tuned in 4.8.
* **`class_weight="balanced"` behaves as expected:** PR-AUC barely changes (0.730 vs 0.736), while at 0.5 recall rises to 0.91 and precision falls to 0.46.
* **Weights:** the largest is `grid_fixed` (-1.60): starting nearer the front means a higher podium probability. Next come the missing-value flags of the qualifying columns (-1.06 and +0.69) and `no_qual_time` (+0.69). They mostly describe the old seasons, where qualifying is missing for everyone, and partly cancel each other, so they should not be read one by one. Then `gap_to_pole_pct` (-0.57), the team's (-0.44) and the driver's (-0.38) championship positions. These are associations the model uses, not causes.
* **Train PR-AUC is much lower than validation** (0.626 vs 0.736): the training score includes the 1950-2005 seasons, which are harder to predict (more cars, more retirements, little qualifying data). It is not a sign of a problem with the model.

**Baseline to beat:** logistic regression, validation PR-AUC **0.736**.

### 4.5 Statistical models: random forest and gradient boosting

Two tree-based models, chosen because they can learn what logistic regression cannot: non-linear effects and interactions between features (for example "a good grid slot matters more for a team that leads the championship"). Both are in scikit-learn.
* **Random forest:** many decision trees, each trained on a random sample of rows and features; the prediction is the average vote. Averaging many different trees reduces overfitting. *Limitation:* its probabilities are averages of votes (not calibrated), and it cannot extrapolate beyond the values seen in training.
* **Gradient boosting** (`HistGradientBoostingClassifier`): small trees built one after another, each one correcting the errors of the trees before it. *Limitation:* more hyperparameters, and it overfits if the trees are too big or too many. scikit-learn's automatic early stopping only starts above 10,000 training rows; we now have 22,441, so it **is** on by default (it holds out 10% of the training rows, never our validation years). To keep the number of trees under our control, we switch it off and set the number of trees ourselves (100 here; Attempt 19 studies it).

**Class imbalance:** both use `class_weight="balanced"` (podiums are about 14% of rows), on top of P3's season weights. Hyperparameters are tuned in 4.7.

#### Attempt 7: random forest
300 trees; `min_samples_leaf=20` so every leaf is based on at least 20 driver-races, which keeps the probabilities smooth instead of memorizing single rows.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = with_preprocessing(RandomForestClassifier(n_estimators=300, min_samples_leaf=20, class_weight="balanced",
                                               n_jobs=-1, random_state=SEED))
rf_results = train_eval(rf, X_train, y_train, X_val, y_val, name="random forest", sample_weight=w_train)
display(rf_results)

#### Attempt 8: gradient boosting
Default tree settings, 100 trees, no automatic early stopping.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = with_preprocessing(HistGradientBoostingClassifier(early_stopping=False, class_weight="balanced",
                                                       random_state=SEED))
gb_results = train_eval(gb, X_train, y_train, X_val, y_val, name="gradient boosting", sample_weight=w_train)
display(gb_results)

#### Two other model families (exploration)
The brief asks for two statistical models, and we have them. To check that we did not miss an easy win, we try two families that work in a completely different way, each with one sensible setting and no tuning. They are not candidates for the final model unless they clearly beat the others.
* **k-nearest neighbours (k-NN):** no training at all. For a new driver-race it finds the 50 most similar driver-races in the training years, and the probability is the share of them that ended on the podium. *Limitation:* every feature counts equally in "similar", and it cannot use training weights. We therefore give it only the full-weight seasons (2006-2019), which is the same as giving the old seasons weight 0.
* **Gaussian Naive Bayes:** for each class it learns a bell curve per feature and multiplies them, assuming the features are **independent** of each other. *Limitation:* our features are strongly linked (grid and qualifying position, points and championship position), so the same evidence is counted several times.

#### Attempt 9: k-nearest neighbours (k = 50, 2006-2019 rows)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = with_preprocessing(KNeighborsClassifier(n_neighbors=50))
knn_results = train_eval(knn, X_train[modern], y_train[modern], X_val, y_val,
                         name="k-nearest neighbours (k=50)")
display(knn_results)

#### Attempt 10: Gaussian Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = with_preprocessing(GaussianNB())
nb_results = train_eval(nb, X_train, y_train, X_val, y_val, name="naive Bayes", sample_weight=w_train)
display(nb_results)

#### Comparison of all models so far
Validation scores for every attempt, plus the train-validation gap in PR-AUC (a large positive gap means overfitting).

In [ ]:
def compare(results):
    """Validation scores of every model, plus the train - validation PR-AUC gap."""
    allres = pd.concat(results, ignore_index=True)
    pr = allres.pivot(index="model", columns="split", values="PR-AUC")
    summary = allres[allres["split"] == "validation"].set_index("model").drop(columns="split")
    summary["PR-AUC gap (train - val)"] = (pr["train"] - pr["validation"]).round(4)
    return summary.sort_values("PR-AUC", ascending=False)

summary = compare([dummy_results, rule_results, lr_results, lr_bal_results, lr_draft_results,
                   rf_results, gb_results, knn_results, nb_results])
display(summary)

lr_pr = summary.loc["logistic regression", "PR-AUC"]
ax = summary["PR-AUC"].sort_values().plot.barh(figsize=(8, 4.5), color="tab:orange")
ax.axvline(lr_pr, color="black", linestyle="--", label=f"baseline to beat (logistic regression) = {lr_pr:.4f}")
ax.set_title("PR-AUC on validation (2020-2021), all models so far")
ax.set_xlabel("PR-AUC (higher is better)")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**What we see (validation 2020-2021).**
* **The tree models tie with logistic regression:** PR-AUC random forest 0.739, logistic regression 0.736, gradient boosting 0.727; ROC-AUC 0.927-0.933.
* **Gradient boosting no longer overfits badly:** train 0.750 vs validation 0.727. With 22,441 training rows (instead of 5,742) the default trees have much more data to learn from.
* **k-NN improved a lot (0.736, now a tie):** in the previous version (19 features) it got 0.677. With P3's shorter list of strong features, "similar driver-race" means similar grid, qualifying and standings, without weak features blurring the distance. A nice side effect of P3's feature selection.
* **Naive Bayes is still the weakest real model (0.570):** better than with 19 features (0.292), because there are fewer duplicate features, but it still counts the same evidence several times (grid, qualifying position, gap and the Q3 flag all say "fast car").
* **Why no gain from trees:** the strongest signal is a smooth "the nearer the front, the likelier the podium" relation that a straight line already captures.

**Decision:** random forest and gradient boosting stay as our two statistical models and are tuned in 4.7. k-NN ties with the baseline but adds nothing over it and cannot use P3's weights; Naive Bayes is far behind. Neither is taken further.

### 4.6 Shallow neural network (PyTorch)

**How it works:** a feed-forward network. Each hidden layer computes weighted sums of its inputs and passes them through a ReLU (negative values become 0), which lets the network learn non-linear patterns. The output layer gives one score that the sigmoid turns into a podium probability. The weights are learned by gradient descent (Adam) on the binary cross-entropy loss. With no hidden layer this would be exactly logistic regression, so the hidden layers are what the network adds.

**Choices** ("shallow" = 1-2 hidden layers, as the brief asks):
* **Dropout 0.2:** during training, 20% of the hidden units are switched off at random in every batch, so the network cannot rely on single units (less overfitting).
* **Early stopping on validation:** after every epoch (one pass over the training rows) we compute the validation loss; training stops when it has not improved for 15 epochs, and the weights of the best epoch are kept.
* **Training weights:** each row's loss is multiplied by its `sample_weight`, so a 1950s row counts a quarter, exactly as in the other models.
* **Class imbalance:** `pos_weight` = (weighted non-podium rows / weighted podium rows) in the loss, the same idea as `class_weight="balanced"`.
* **Preprocessing:** the same pipeline as the other models (4.2), fitted on the training rows only inside `fit()`.
* **Reproducible:** `torch.manual_seed(SEED)`, a seeded batch order and a single CPU thread, so Run All gives the same numbers on a laptop and on Kaggle.

*Limitations:* many settings, needs scaling and imputation, a black box without XAI, and on small tables it rarely beats simpler models. **Caveat:** the validation set chooses the stopping epoch, so the network's validation score is slightly optimistic compared with the other models; the test set gives the fair comparison.

In [ ]:
%pip install torch

import torch
from torch import nn


class ShallowNN:
    """Feed-forward network wrapped with fit / predict_proba so train_eval() can score it.
    - hidden: the units of each hidden layer, e.g. (32,) = one layer, (64, 32) = two layers
    - preprocessing: the shared pipeline of 4.2, fitted on the training rows only, inside fit()
    - sample_weight: each training row's loss is multiplied by its weight
    - early stopping: training stops when the validation loss has not improved for `patience` epochs,
      and the weights of the best epoch are kept"""

    def __init__(self, X_val, y_val, hidden=(32,), dropout=0.2, lr=1e-3, batch_size=256,
                 max_epochs=200, patience=15, seed=SEED):
        self.X_val, self.y_val = X_val, y_val
        self.hidden, self.dropout, self.lr, self.batch_size = hidden, dropout, lr, batch_size
        self.max_epochs, self.patience, self.seed = max_epochs, patience, seed

    def _build(self, n_inputs):
        layers, size = [], n_inputs
        for h in self.hidden:                                   # hidden layer: linear -> ReLU -> dropout
            layers += [nn.Linear(size, h), nn.ReLU(), nn.Dropout(self.dropout)]
            size = h
        layers.append(nn.Linear(size, 1))                       # output: one score (logit) for "podium"
        return nn.Sequential(*layers)

    def _tensor(self, X):
        return torch.tensor(self.prep.transform(X), dtype=torch.float32)

    def fit(self, X, y, sample_weight=None):
        torch.manual_seed(self.seed)
        torch.set_num_threads(1)                                # same numbers on every machine
        self.prep = clone(preprocess).fit(X)
        Xt, yt = self._tensor(X), torch.tensor(np.asarray(y), dtype=torch.float32)
        wt = torch.ones(len(yt)) if sample_weight is None else torch.tensor(np.asarray(sample_weight), dtype=torch.float32)
        Xv, yv = self._tensor(self.X_val), torch.tensor(np.asarray(self.y_val), dtype=torch.float32)

        self.net = self._build(Xt.shape[1])
        self.n_weights = sum(p.numel() for p in self.net.parameters())
        pos_weight = wt[yt == 0].sum() / wt[yt == 1].sum()      # class imbalance, like class_weight="balanced"
        row_loss = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")   # one loss per row
        optimizer = torch.optim.Adam(self.net.parameters(), lr=self.lr)
        gen = torch.Generator().manual_seed(self.seed)

        self.history = {"train": [], "validation": []}
        best_loss, best_state, waited = float("inf"), None, 0
        for epoch in range(self.max_epochs):
            self.net.train()                                   # dropout on
            batch_losses = []
            for idx in torch.randperm(len(Xt), generator=gen).split(self.batch_size):
                optimizer.zero_grad()
                losses = row_loss(self.net(Xt[idx]).squeeze(1), yt[idx])
                loss = (losses * wt[idx]).sum() / wt[idx].sum()  # weighted mean: old seasons count less
                loss.backward()
                optimizer.step()
                batch_losses.append(loss.item())
            self.net.eval()                                    # dropout off
            with torch.no_grad():
                val_loss = row_loss(self.net(Xv).squeeze(1), yv).mean().item()
            self.history["train"].append(float(np.mean(batch_losses)))
            self.history["validation"].append(val_loss)

            if val_loss < best_loss:                           # early stopping on the validation loss
                best_loss, waited = val_loss, 0
                best_state = {k: v.clone() for k, v in self.net.state_dict().items()}
                self.best_epoch = epoch + 1
            else:
                waited += 1
                if waited >= self.patience:
                    break
        self.net.load_state_dict(best_state)                   # keep the best epoch, not the last one
        return self

    def predict_proba(self, X):
        self.net.eval()
        with torch.no_grad():
            p = torch.sigmoid(self.net(self._tensor(X)).squeeze(1)).numpy()
        return np.column_stack([1 - p, p])


def run_network(hidden, name):
    """Train one network on the training years (with the weights), score it, print its size and stopping epoch."""
    net = ShallowNN(X_val, y_val, hidden=hidden)
    res = train_eval(net, X_train, y_train, X_val, y_val, name=name, sample_weight=w_train)
    display(res)
    print(f"{net.n_weights} weights; stopped after {len(net.history['train'])} epochs, best epoch: {net.best_epoch}")
    return net, res

#### Attempt 11: one hidden layer (32 units)

In [ ]:
nn1, nn1_results = run_network((32,), "neural network (32)")

#### Attempt 12: two hidden layers (64 and 32 units)

In [ ]:
nn2, nn2_results = run_network((64, 32), "neural network (64-32)")

#### Attempt 13: a wider layer (one hidden layer, 128 units)
Same depth as Attempt 11 but four times the units. If the 32-unit layer were too small to hold the patterns, more units would help.

In [ ]:
nn128, nn128_results = run_network((128,), "neural network (128)")

#### Attempt 14: a deeper network (three hidden layers, 128-64-32)
One step beyond "shallow": if 1 and 2 layers are close, does a third layer find anything more? (Kept out of the tuning grid in 4.7, because the brief asks for a shallow network; this attempt only shows what depth does.)

In [ ]:
nn3, nn3_results = run_network((128, 64, 32), "neural network (128-64-32)")

#### Loss curves
Training loss (weighted average over the batches of each epoch, with dropout on) and validation loss per epoch. The dashed line marks the epoch whose weights were kept.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for ax, net, title in [(axes[0, 0], nn1, "1 hidden layer (32)"), (axes[0, 1], nn2, "2 hidden layers (64-32)"),
                       (axes[1, 0], nn128, "1 wide hidden layer (128)"), (axes[1, 1], nn3, "3 hidden layers (128-64-32)")]:
    epochs = range(1, len(net.history["train"]) + 1)
    ax.plot(epochs, net.history["train"], label="training loss")
    ax.plot(epochs, net.history["validation"], label="validation loss")
    ax.axvline(net.best_epoch, color="black", linestyle="--", label=f"best epoch ({net.best_epoch})")
    ax.set_title(f"Neural network, {title}")
    ax.set_xlabel("epoch")
    ax.legend()
for ax in axes[:, 0]:
    ax.set_ylabel("weighted binary cross-entropy")
plt.tight_layout()
plt.show()

#### Comparison of the networks (validation)

In [ ]:
nets = compare([nn1_results, nn2_results, nn128_results, nn3_results])
sizes = {"neural network (32)": nn1, "neural network (64-32)": nn2, "neural network (128)": nn128,
         "neural network (128-64-32)": nn3}
nets.insert(0, "weights", [sizes[m].n_weights for m in nets.index])
display(nets)

**What we see (validation 2020-2021).**
* **All four networks are a little above logistic regression (0.736):** 0.759 with one layer of 32, 0.759 with two layers (64-32), **0.763 with one wide layer of 128**, 0.755 with three layers.
* **Wider (Attempt 13):** this time the 128-unit layer is the best network, but only by 0.005, with four times the weights of the 32-unit layer (2,305 vs 577). That is far below the noise of the validation PR-AUC (about 0.037), so it is a tie, not a win.
* **Deeper (Attempt 14):** the third layer gives 0.755, no better, with the most weights (12,545). It reaches its best validation loss already at **epoch 8**, then the validation loss creeps up while the training loss keeps falling: the start of overfitting.
* **All four are within 0.008 of each other.** So the honest answer to "why not a bigger or deeper network?" is: we tried; with this table, size does not matter, and a shallow network is enough.
* **Loss curves:** both losses fall quickly in the first 5-10 epochs and then flatten. This time the **training** loss is above the validation loss, because the training rows include the old seasons, which are harder to predict (the loss is weighted, but those rows are still there).
* **At threshold 0.5** the networks behave like the other balanced models: high recall (0.90-0.91), precision about 0.50.

### 4.7 Hyperparameter tuning (validation only)

For each model we try a small grid of the settings that control its complexity. Every combination is trained on the training years (with the weights) and scored on validation with `train_eval()`; the best combination is the one with the highest **validation PR-AUC** (our primary metric, see 4.8). The test years are not touched.

The grids are deliberately small: the validation set has only 117 podiums, so a large search would mostly fit its noise.

In [ ]:
from sklearn.model_selection import ParameterGrid

def tune(name, make_model, grid):
    """Train one model per combination in `grid` (training years, with the weights) and score it on validation.
    Returns the fitted model with the best validation PR-AUC and the table of all results."""
    rows, best_model, best_pr = [], None, -1
    for params in ParameterGrid(grid):
        candidate = make_model(**params)
        res = train_eval(candidate, X_train, y_train, X_val, y_val, name=name,
                         sample_weight=w_train).set_index("split")
        rows.append({**params, "train PR-AUC": res.loc["train", "PR-AUC"],
                     "val PR-AUC": res.loc["validation", "PR-AUC"], "val ROC-AUC": res.loc["validation", "ROC-AUC"]})
        if res.loc["validation", "PR-AUC"] > best_pr:
            best_model, best_pr = candidate, res.loc["validation", "PR-AUC"]
    return best_model, pd.DataFrame(rows).sort_values("val PR-AUC", ascending=False, ignore_index=True)

tuned = {}   # model name -> (best fitted model, table of all combinations)

#### Attempt 15: tuning logistic regression
`C` is the inverse regularization strength: a small `C` pushes the weights towards 0 (simpler model), a large `C` lets them grow.

In [ ]:
tuned["logistic regression"] = tune(
    "logistic regression",
    lambda C: with_preprocessing(LogisticRegression(C=C, max_iter=2000, random_state=SEED)),
    {"C": [0.01, 0.1, 1, 10]})
display(tuned["logistic regression"][1])

#### Attempt 16: tuning the random forest
`min_samples_leaf`: the minimum number of driver-races behind each leaf (larger = smoother, less memorizing). `max_features`: how many inputs each split may look at (`"sqrt"` = a random subset, `1.0` = all of them).

In [ ]:
tuned["random forest"] = tune(
    "random forest",
    lambda min_samples_leaf, max_features: with_preprocessing(
        RandomForestClassifier(n_estimators=300, min_samples_leaf=min_samples_leaf, max_features=max_features,
                               class_weight="balanced", n_jobs=-1, random_state=SEED)),
    {"min_samples_leaf": [5, 20, 50], "max_features": ["sqrt", 1.0]})
display(tuned["random forest"][1])

#### Attempt 17: tuning gradient boosting
`learning_rate`: how much each new tree may correct the previous ones (smaller = slower, more careful). `max_leaf_nodes`: the size of each tree (smaller = simpler trees). 100 trees each.

In [ ]:
tuned["gradient boosting"] = tune(
    "gradient boosting",
    lambda learning_rate, max_leaf_nodes: with_preprocessing(HistGradientBoostingClassifier(
        learning_rate=learning_rate, max_leaf_nodes=max_leaf_nodes, early_stopping=False,
        class_weight="balanced", random_state=SEED)),
    {"learning_rate": [0.03, 0.1], "max_leaf_nodes": [7, 15, 31]})
display(tuned["gradient boosting"][1])

#### Attempt 18: tuning the neural network
`hidden`: the size of the hidden layer(s), including the 128-unit layer of Attempt 13. `dropout`: the share of hidden units switched off during training. (The number of epochs is already chosen by early stopping.)

In [ ]:
tuned["neural network"] = tune(
    "neural network",
    lambda hidden, dropout: ShallowNN(X_val, y_val, hidden=hidden, dropout=dropout),
    {"hidden": [(16,), (32,), (128,), (64, 32)], "dropout": [0.2, 0.4]})
display(tuned["neural network"][1])

**What we see (tuning, validation PR-AUC).**
* **Logistic regression:** `C` hardly matters (0.729-0.737); best `C=10` (0.737). With 12 features and 22,441 rows the model is not short of data.
* **Random forest:** best with `max_features="sqrt"` and `min_samples_leaf=5` (0.740), just above `min_samples_leaf=20` (0.739). But small leaves memorize: train 0.842 vs validation 0.740. Letting every split see all features (`1.0`) is worse (0.711-0.731).
* **Gradient boosting:** best with small trees and the faster rate (`max_leaf_nodes=7`, `learning_rate=0.1`): 0.733. The biggest trees are the worst on validation.
* **Neural network:** best with one hidden layer of 128 and dropout 0.2: 0.763 (Attempt 13). The 128-unit layer is also second with more dropout (0.760). Two layers 0.759, 32 units 0.759.
* **Tuning gains are small** (at most +0.005, for gradient boosting), and the four tuned models end within 0.031 of each other (0.733-0.763), less than one standard error.

#### Attempt 19: how many trees? (random forest and gradient boosting)
Until now the forest always had 300 trees and boosting 100. Here we keep each model's tuned settings and change only the number of trees, scoring train and validation every time. The two families should react differently: a forest **averages** independent trees, so more trees can only make the average steadier; boosting **adds** trees that keep correcting the training errors, so too many trees start to learn the noise of the training years.

In [ ]:
best_rf = tuned["random forest"][1].iloc[0]          # tuned settings from Attempts 16 and 17
best_gb = tuned["gradient boosting"][1].iloc[0]
tree_counts = [10, 25, 50, 100, 200, 400, 800]

rows = []
for n in tree_counts:
    forest = with_preprocessing(RandomForestClassifier(
        n_estimators=n, min_samples_leaf=int(best_rf["min_samples_leaf"]), max_features=best_rf["max_features"],
        class_weight="balanced", n_jobs=-1, random_state=SEED))
    boosting = with_preprocessing(HistGradientBoostingClassifier(
        max_iter=n, early_stopping=False, learning_rate=best_gb["learning_rate"],
        max_leaf_nodes=int(best_gb["max_leaf_nodes"]), class_weight="balanced", random_state=SEED))
    for family, clf in [("random forest", forest), ("gradient boosting", boosting)]:
        res = train_eval(clf, X_train, y_train, X_val, y_val, name=family, sample_weight=w_train).set_index("split")
        rows.append({"model": family, "trees": n, "train PR-AUC": res.loc["train", "PR-AUC"],
                     "val PR-AUC": res.loc["validation", "PR-AUC"]})
trees = pd.DataFrame(rows)
display(trees.pivot(index="trees", columns="model", values=["train PR-AUC", "val PR-AUC"]).round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, family, used in [(axes[0], "random forest", 300), (axes[1], "gradient boosting", 100)]:
    part = trees[trees["model"] == family]
    ax.plot(part["trees"], part["train PR-AUC"], marker="o", label="train")
    ax.plot(part["trees"], part["val PR-AUC"], marker="o", label="validation")
    ax.axvline(used, color="black", linestyle="--", label=f"number used in the models ({used})")
    ax.set_xscale("log")
    ax.set_xticks(tree_counts, [str(n) for n in tree_counts])
    ax.set_title(f"{family}: PR-AUC vs number of trees")
    ax.set_xlabel("number of trees (log scale)")
    ax.legend()
axes[0].set_ylabel("PR-AUC")
plt.tight_layout()
plt.show()

**What we see (number of trees).**
* **Random forest: more trees never hurt, they only steady the average.** Validation PR-AUC rises from 0.707 with 10 trees to 0.739 with 200 and stays there (0.740 with 800); the training score is flat (about 0.84). The 300 trees we use are on the plateau.
* **Gradient boosting: the number of trees is a complexity knob.** The training score keeps rising (0.611 with 10 trees to 0.783 with 800) while validation peaks at 200 trees (0.736) and then falls clearly (0.731 at 400, **0.683 at 800**): every extra tree corrects the remaining training errors, and after a point those errors are noise.
* **We keep the tuned models unchanged.** 200 boosting trees would add +0.004 on validation, far below the noise, and it would not change the final model (4.9). The lesson is the shape: for boosting the number of trees must be limited (or stopped early), for a forest it does not matter.

#### Attempt 20: polynomial features for logistic regression (exploration)
Logistic regression is linear (4.4). Instead of a new model family, we can give it non-linear inputs: `PolynomialFeatures` adds every square and every product of two inputs (degree 2, for example grid x team standing), or of up to three (degree 3). The 16 inputs (12 features + 4 missing-value flags) become 152 or 968. More inputs means more complexity, so each degree is combined with stronger regularization (smaller `C`). It is an exploration: it is not added to `tuned` and takes no part in the choice of the final model.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures


def poly_lr(degree, C):
    """Logistic regression on polynomial inputs: the shared preprocessing, then every product of up to `degree`
    inputs (degree 2 = squares and pairs, e.g. grid x team standing), scaled again, then the model."""
    return Pipeline([("prep", clone(preprocess)),
                     ("poly", PolynomialFeatures(degree, include_bias=False)),
                     ("scale", StandardScaler()),
                     ("model", LogisticRegression(C=C, max_iter=5000, random_state=SEED))])

# an exploration: not added to `tuned`, so it takes no part in the choice of the final model (4.9)
_, poly_table = tune("logistic regression (polynomial)", poly_lr, {"degree": [1, 2, 3], "C": [0.01, 0.1, 1]})
n_inputs = {d: poly_lr(d, 1)[:2].fit(X_train).named_steps["poly"].n_output_features_ for d in (1, 2, 3)}
poly_table.insert(2, "inputs", poly_table["degree"].map(n_inputs))
display(poly_table)

**What we see (polynomial features, validation PR-AUC).**
* **Degree 2 helps a little, and only with strong regularization:** best 0.755 (`C=0.01`) vs 0.736 for the plain model (+0.018). Without it (`C=1`) degree 2 is worse (0.720).
* **Degree 3 overfits:** with 968 inputs the training score rises (0.668 at `C=1`) while validation falls (0.682). Only heavy regularization brings it back to 0.728.
* **The gain is below the noise:** +0.018 is about half of one standard error of the validation PR-AUC (about 0.037, 4.9), and `C` was chosen on validation, so the number is slightly optimistic. It is in line with the trees and the networks: the non-linear patterns add at most about 0.02.
* **The plain logistic regression stays in the comparison:** the main signal is a smooth "the nearer the front, the likelier the podium" relation that a straight line already captures.

### 4.8 Metric choice and decision threshold

**Primary metric: PR-AUC.** Podiums are rare (about 15% of the validation rows). PR-AUC measures how well the real podium finishers are ranked at the top, it does not depend on a threshold, and its floor is the podium rate (the dummy in 4.3) instead of 0.5, so it shows clearly how much a model adds.

**Secondary metric: F1 at the chosen threshold.** In the end the model must say yes or no; F1 balances precision (how many predicted podiums are right) and recall (how many real podiums are found).

ROC-AUC is reported for completeness, but with about 85% non-podium rows it looks good easily (about 0.8 even for the simple rule).

**Threshold:** for every tuned model we try thresholds from 0.05 to 0.95 (step 0.01) on validation and keep the one with the highest F1. Each model gets its own threshold, because class weighting shifts the probabilities (4.4).

In [ ]:
def best_threshold(clf, X, y):
    """Threshold between 0.05 and 0.95 (step 0.01) with the highest F1 on (X, y)."""
    proba = clf.predict_proba(X)[:, 1]
    candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
    f1s = [f1_score(y, proba >= t, zero_division=0) for t in candidates]
    return candidates[int(np.argmax(f1s))]

rows = []
for name, (clf, _) in tuned.items():
    t = best_threshold(clf, X_val, y_val)
    rows.append({"model": name, "threshold": t, **evaluate(clf, X_val, y_val, threshold=t)})
tuned_table = pd.DataFrame(rows).set_index("model").round(4)
thresholds = tuned_table["threshold"].to_dict()
display(tuned_table)

### 4.9 Choosing the final model (validation only)

Before comparing the models we measure how noisy the validation PR-AUC is: we resample the validation rows with replacement 1000 times (bootstrap) and take the standard deviation of PR-AUC as its standard error (SE).

**Rule (one-standard-error rule):** among the models whose validation PR-AUC is within one SE of the best, take the simplest (order: logistic regression < random forest < gradient boosting < neural network). A difference smaller than the noise is not a reason to accept a more complex model. The final model and its threshold are then used once on the test set (4.10) and by P1's inference function (4.13).

In [ ]:
COMPLEXITY = ["logistic regression", "random forest", "gradient boosting", "neural network"]   # simplest first
val_pr = tuned_table["PR-AUC"]
best_name = val_pr.idxmax()

# bootstrap standard error of the best model's validation PR-AUC
rng = np.random.default_rng(SEED)
proba, y = tuned[best_name][0].predict_proba(X_val)[:, 1], y_val.to_numpy()
boot = []
for _ in range(1000):
    i = rng.integers(0, len(y), len(y))
    boot.append(average_precision_score(y[i], proba[i]))
se = np.std(boot)

within_one_se = [name for name in COMPLEXITY if val_pr[name] >= val_pr[best_name] - se]
FINAL_NAME = within_one_se[0]
FINAL_MODEL, FINAL_THRESHOLD = tuned[FINAL_NAME][0], thresholds[FINAL_NAME]

print(f"best validation PR-AUC: {best_name} = {val_pr[best_name]:.4f}, bootstrap SE = {se:.4f} "
      f"(95% range {np.percentile(boot, 2.5):.3f}-{np.percentile(boot, 97.5):.3f})")
print(f"models within one SE: {within_one_se}")
print(f"final model: {FINAL_NAME}, threshold {FINAL_THRESHOLD}")

**How the final threshold was chosen:** precision, recall and F1 of the final model on validation for every threshold.

In [ ]:
proba = FINAL_MODEL.predict_proba(X_val)[:, 1]
candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
curve = pd.DataFrame([evaluate(FINAL_MODEL, X_val, y_val, threshold=t) for t in candidates], index=candidates)

ax = curve[["precision", "recall", "F1"]].plot(figsize=(8, 4))
ax.axvline(FINAL_THRESHOLD, color="black", linestyle="--", label=f"chosen threshold = {FINAL_THRESHOLD}")
ax.set_title(f"Final model ({FINAL_NAME}): metrics vs threshold on validation")
ax.set_xlabel("threshold (predict podium if probability >= threshold)")
ax.set_ylabel("score")
ax.set_ylim(0, 1)
ax.legend(loc="center right")
plt.tight_layout()
plt.show()

# compare the final yes/no decisions with the simple rule grid <= 3 from 4.4
rule_pred = X_val["grid_fixed"].between(1, 3).to_numpy()
disagree = int(((proba >= FINAL_THRESHOLD) != rule_pred).sum())
print(f"rows where the final model and the rule grid <= 3 disagree: {disagree} of {len(X_val)}")
display(curve.loc[sorted({0.2, 0.3, 0.4, FINAL_THRESHOLD, 0.5, 0.6})].round(3))     # how flat F1 is around the choice

**What we see (threshold and final model).**
* **Each model needs its own threshold:** 0.29 for logistic regression, 0.69 for the random forest, 0.71 for gradient boosting, 0.72 for the network. The balanced models have their probabilities pushed up by the class weights, so their best threshold is higher.
* **With its own threshold every model reaches a validation F1 of about 0.71-0.74** (network 0.736, gradient boosting 0.722, random forest 0.718, logistic regression 0.710).
* **Final model:** the bootstrap standard error of the validation PR-AUC is 0.037 (95% range 0.69-0.83). All four models are within one standard error of the best (the network, 0.763), so the one-standard-error rule picks the simplest: **logistic regression (`C=10`) with threshold 0.29**.
* **The threshold sits on a plateau:** validation F1 is 0.660 at 0.20, **0.710 at 0.29**, 0.708 at 0.30, 0.687 at 0.40 and 0.641 at 0.50. Anything between about 0.25 and 0.35 is almost equally good, which is why the exact value moves between table versions (0.23, then 0.42, now 0.29) while F1 barely changes.
* **More than the rule:** the final model disagrees with the rule `grid <= 3` on 43 of 774 validation rows.

### 4.10 Final evaluation on the test set (used once)

Every model, hyperparameter, threshold and the final choice were fixed on the training and validation years. Here each tuned model is scored **once** on the test set (2022-2024) with the threshold it got on validation, next to its train and validation scores. Nothing is changed after this point. The rule `grid <= 3` is the reference to beat.

**Honest note on the test set.** This part was run on four versions of P3's table: (1) 18 features with a one-hot `era`, where the test probabilities were pushed up because the test era never occurs in training; (2) 17 features from 2003; (3) 20 features from 2006, of which we used 19; (4) this run: P3's final 12 features, training 1950-2019 with weights. Every change of the table was made by P3 with reasons documented on training and validation years (her ablation never uses the test years). **Nothing in this part was chosen on the test set:** the models, the hyperparameter grids, the threshold rule and the final-model rule are the same in every run, and every choice is made on validation.

In [ ]:
test = table[table["split"] == "test"]
X_test, y_test = test[FEATURES].astype(float), test["podium"]
assert (test["sample_weight"] == 1).all()
print(f"test: {len(X_test)} rows ({test['year'].min()}-{test['year'].max()}), {y_test.sum()} podiums, "
      f"podium rate {y_test.mean():.3f} (= the PR-AUC of a model that knows nothing)")

final_models = {"rule grid <= 3": (GridRule(), 0.5),
                **{name: (clf, thresholds[name]) for name, (clf, _) in tuned.items()}}
rows = []
for name, (clf, t) in final_models.items():
    label = f"{name} (final)" if name == FINAL_NAME else name
    for split, X, y, w in [("train", X_train, y_train, w_train), ("validation", X_val, y_val, None),
                           ("test", X_test, y_test, None)]:
        rows.append({"model": label, "split": split, "threshold": t, **evaluate(clf, X, y, t, w)})
comparison = pd.DataFrame(rows).round(4)
display(comparison)

# the two metrics we chose (4.8), side by side per split
display(comparison.pivot(index="model", columns="split", values=["PR-AUC", "F1"])
        .reindex(columns=["train", "validation", "test"], level=1))

#### ROC and precision-recall curves (test set)
The ROC curve shows the true-positive rate against the false-positive rate for every threshold; the precision-recall curve shows precision against recall. The area under each curve is the ROC-AUC and the PR-AUC (`AP`) in the table.

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5))
for name, (clf, _) in tuned.items():
    proba = clf.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax_roc)
    PrecisionRecallDisplay.from_predictions(y_test, proba, name=name, ax=ax_pr)
ax_roc.plot([0, 1], [0, 1], color="black", linestyle="--", label="random guess (AUC = 0.5)")
ax_pr.axhline(y_test.mean(), color="black", linestyle="--", label=f"podium rate = {y_test.mean():.3f}")
ax_roc.set_title("ROC curves on the test set (2022-2024)")
ax_pr.set_title("Precision-recall curves on the test set (2022-2024)")
ax_roc.legend(loc="lower right")
ax_pr.legend(loc="lower left")
plt.tight_layout()
plt.show()

#### Confusion matrices (test set, threshold chosen on validation)
Rows are the real outcome, columns the prediction. Top-right = false alarms (predicted a podium that did not happen), bottom-left = missed podiums.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, (clf, _)) in zip(axes, tuned.items()):
    pred = (clf.predict_proba(X_test)[:, 1] >= thresholds[name]).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["no podium", "podium"],
                                            ax=ax, colorbar=False)
    ax.set_title(f"{name}{' (final)' if name == FINAL_NAME else ''}\nthreshold {thresholds[name]}")
plt.tight_layout()
plt.show()

#### Final comparison plot (train, validation, test)
The two metrics we chose (4.8) for every model on each split, in one picture. The dashed line in the PR-AUC panel is the no-skill floor on test (the test podium rate). A model that generalizes keeps its bars from validation to test; a big drop means it learned the training or validation years too well.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
model_order = list(dict.fromkeys(comparison["model"]))[::-1]          # rule at the top, as in the table
for ax, metric in zip(axes, ["PR-AUC", "F1"]):
    scores = comparison.pivot(index="model", columns="split", values=metric).loc[model_order]
    scores[["train", "validation", "test"]].plot.barh(ax=ax, color=["tab:gray", "tab:orange", "tab:blue"])
    ax.set_title(f"{metric} per model and split (threshold chosen on validation)")
    ax.set_xlabel(f"{metric} (higher is better)")
    ax.set_xlim(0, 1)
axes[0].axvline(y_test.mean(), color="black", linestyle="--", label=f"no-skill floor on test = {y_test.mean():.3f}")
axes[0].legend(loc="upper right")
axes[0].set_ylabel("")
plt.tight_layout()
plt.show()

**What we see (final comparison plot).**
* **Every model is far above the no-skill floor on test (0.151).** The rule `grid <= 3` is clearly behind in PR-AUC (0.474 against 0.67-0.70 for the four models) but only a little behind in F1 (0.649 against 0.66-0.68), because it gives only yes/no and cannot rank.
* **From validation to test, logistic regression drops least** (0.737 to 0.692); the random forest (0.741 to 0.669) and the network (about 0.76 to 0.69) drop most. This is the picture behind the one-standard-error rule of 4.9.
* **The random forest is the only model whose training bar is far above the others** (train PR-AUC 0.843): its small leaves memorized the training seasons. For the other models the training bar is *below* validation, because the training years include the harder 1950-2005 seasons (4.4), not because of a problem.
* **On test the four models are bunched together** (PR-AUC 0.67-0.70, F1 0.66-0.68), so choosing the simplest one cost nothing.

**What we see (test set 2022-2024, 1,351 rows, 204 podiums).**
* **Ranking on unseen seasons:** PR-AUC network 0.693, logistic regression 0.692 (a tie), gradient boosting 0.682, random forest 0.670; rule 0.474; no-skill floor 0.151. ROC-AUC is 0.93 for all four.
* **Yes/no decisions:** F1 network 0.685, logistic regression 0.676, gradient boosting 0.667, random forest 0.656, rule 0.649. The network's lead (+0.009) is well inside the noise, and switching to it now would be a choice made on the test set, so we do not.
* **The one-standard-error rule held again:** from validation to test the network drops most (0.763 to 0.693, its stopping epoch was chosen on validation) and the forest drops a lot (0.740 to 0.670, its small leaves memorized the training seasons: train 0.842), while the simplest model drops least (0.737 to 0.692).
* **Final model in numbers (threshold 0.29):** it finds 169 of the 204 podiums (recall 0.83) with 127 false alarms (precision 0.57) and 35 missed podiums. The lower threshold favours finding podiums.
* **Compared with the previous run (19 features, 2006-2019, threshold 0.42):** test PR-AUC 0.702 then, 0.692 now; F1 0.673 then, 0.676 now. Both differences are well inside the noise: P3's shorter feature list and the weighted old seasons give practically the same model quality, with 12 features instead of 19.
* **No choice was made on the test set:** the final model and threshold are the ones chosen on validation in 4.9.

### 4.11 Explainability (XAI) of the final model

We explain the final model chosen in 4.9 (`FINAL_MODEL`). The code only calls `predict_proba`, so it works for any of our models. We use the validation years: explanations do not choose anything, so there is no reason to touch the test set again.

* **Permutation importance (global):** shuffle one feature at a time and measure how much the validation PR-AUC drops. A big drop means the model relies on that feature. Because several features carry the same information (for example `grid_fixed` and `qual_position`), we also shuffle each of P3's **feature groups** together: shuffling only one of two twin features underestimates both, since the model can lean on the twin.
* **SHAP (global and local):** splits each prediction into one contribution per feature, starting from a baseline (the average prediction). The contributions add up exactly to the predicted probability.
* **LIME (local):** for one row, creates many slightly changed copies, asks the model about them, and fits a small linear model around that row. Its weights say which feature values push this one prediction up or down.

**Not causal.** All three methods describe how *the model* uses the features to make predictions. A feature being important is not proof that it causes podium finishes: starting at the front does not by itself put a driver on the podium; a fast car and driver cause both the good grid slot and the podium, and the model only uses the grid slot as a signal of that.

#### Global: permutation importance (validation, 30 shuffles)

In [ ]:
from sklearn.inspection import permutation_importance

pr_auc_scorer = lambda clf, X, y: average_precision_score(y, clf.predict_proba(X)[:, 1])   # works for any of our models
pi = permutation_importance(FINAL_MODEL, X_val, y_val, scoring=pr_auc_scorer, n_repeats=30, random_state=SEED)
perm = pd.DataFrame({"PR-AUC drop": pi.importances_mean, "std": pi.importances_std}, index=FEATURES)

# the same idea per feature group: shuffle all columns of a group together
rng = np.random.default_rng(SEED)
full_score = pr_auc_scorer(FINAL_MODEL, X_val, y_val)
group_rows = []
for group, cols in FEATURE_GROUPS.items():
    drops = []
    for _ in range(30):
        shuffled = X_val.copy()
        shuffled[cols] = X_val[cols].iloc[rng.permutation(len(X_val))].set_axis(X_val.index)
        drops.append(full_score - pr_auc_scorer(FINAL_MODEL, shuffled, y_val))
    group_rows.append({"group": group, "PR-AUC drop": np.mean(drops), "std": np.std(drops)})
group_perm = pd.DataFrame(group_rows).set_index("group")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [3, 2]})
for ax, imp, title in [(ax1, perm, "per feature"), (ax2, group_perm, "per feature group (shuffled together)")]:
    imp = imp.sort_values("PR-AUC drop")
    ax.barh(imp.index, imp["PR-AUC drop"], xerr=imp["std"], color="tab:blue")
    ax.set_title(f"Permutation importance {title}")
    ax.set_xlabel("drop in validation PR-AUC (bar = mean, line = std)")
fig.suptitle(f"Final model: {FINAL_NAME}")
plt.tight_layout()
plt.show()
display(perm.sort_values("PR-AUC drop", ascending=False).round(4).head(8))
display(group_perm.sort_values("PR-AUC drop", ascending=False).round(4))

#### Global: SHAP summary (validation)
SHAP and LIME call the model with plain number arrays, so a small helper `predict_podium_numeric` first turns them back into a table with the feature names. (Its name differs from P1's inference function `predict_podium` on purpose, so the two do not overwrite each other in the merged notebook.) Rows with a missing value (mostly round 1, without a championship position) are left out of the explanations, so every explained value is real. Each dot is one validation row. Position = how much that feature moved the predicted podium probability for that row (right = up, left = down); colour = the feature's value (red = high, blue = low).

In [ ]:
import warnings
with warnings.catch_warnings():          # shap's import prints a harmless progress-bar warning in notebooks
    warnings.simplefilter("ignore")
    import shap

def predict_podium_numeric(A):
    """Podium probability for a plain number array in FEATURES order (how SHAP and LIME call the model)."""
    return FINAL_MODEL.predict_proba(pd.DataFrame(np.asarray(A, dtype=float), columns=FEATURES))[:, 1]

complete = val[FEATURES].notna().all(axis=1)
xai_rows = val[complete]                                   # validation rows without missing values
X_xai, y_xai = xai_rows[FEATURES].astype(float), xai_rows["podium"].to_numpy()
print(f"explained rows: {len(X_xai)} of {len(val)} validation rows")

background = X_xai.sample(100, random_state=SEED)          # SHAP's baseline = average prediction over these rows
explainer = shap.Explainer(predict_podium_numeric, shap.maskers.Independent(background), algorithm="permutation",
                           feature_names=FEATURES, seed=SEED)
shap_values = explainer(X_xai, max_evals=2 * len(FEATURES) + 1, silent=True)
print(f"SHAP baseline (average predicted podium probability of the background rows): {shap_values.base_values[0]:.3f}")

shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.title(f"SHAP summary of the final model ({FINAL_NAME}), validation")
plt.xlabel("SHAP value (change in predicted podium probability)")
plt.tight_layout()
plt.show()

#### Local: one podium row and one non-podium row
Chosen by a fixed rule, not by hand: the podium finisher with the **highest** predicted probability (a clear case), and the non-podium row with the **median** predicted probability among non-podiums (a typical midfield car). The finishing position and status are shown only to tell the story; they are post-race information and never features.

In [ ]:
race_names = tables["races"].set_index("raceId")["name"]
driver_names = tables["drivers"].set_index("driverId")["surname"]
finish = outcomes.set_index(["raceId", "driverId"])[["positionOrder", "status"]]

proba_xai = predict_podium_numeric(X_xai)
non_podium_rows = np.where(y_xai == 0)[0]
examples = {
    "podium": int(np.argmax(np.where(y_xai == 1, proba_xai, -1))),
    "non-podium": int(non_podium_rows[np.argsort(proba_xai[non_podium_rows])[len(non_podium_rows) // 2]]),
}

def describe(i):
    """Who/where for explained row i (finishing position and status from `outcomes`, for the story only)."""
    r = xai_rows.iloc[i]
    pos, status = finish.loc[(r["raceId"], r["driverId"])]
    return (f"{driver_names[r['driverId']]}, {race_names[r['raceId']]} {r['year']}: grid {r['grid']}, "
            f"predicted {proba_xai[i]:.1%}, finished P{pos} ({status})")

for label, i in examples.items():
    print(f"{label:>10}: {describe(i)}")

**Local SHAP (waterfall):** starts at the baseline at the bottom and adds each feature's contribution until it reaches this row's predicted probability at the top.

In [ ]:
for label, i in examples.items():
    shap.plots.waterfall(shap_values[i], max_display=10, show=False)
    plt.title(f"Local SHAP, {label} example\n{describe(i)}")
    plt.tight_layout()
    plt.show()

**LIME:** the bars are the weights of LIME's small linear model around this row (green pushes towards podium, red away from it). We also print how close LIME's local model gets to the real model's probability.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

yes_no_columns = [j for j, c in enumerate(FEATURES) if X_xai[c].nunique() <= 2]   # 0/1 features, treated as categories
lime_explainer = LimeTabularExplainer(X_xai.to_numpy(), feature_names=FEATURES,
                                      class_names=["no podium", "podium"], categorical_features=yes_no_columns,
                                      mode="classification", random_state=SEED)
predict_both = lambda A: np.column_stack([1 - predict_podium_numeric(A), predict_podium_numeric(A)])

for label, i in examples.items():
    lime_result = lime_explainer.explain_instance(X_xai.iloc[i].to_numpy(), predict_both, num_features=10)
    lime_result.as_pyplot_figure()
    plt.title(f"LIME, {label} example\n{describe(i)}")
    plt.xlabel("LIME weight (effect on the podium probability)")
    plt.tight_layout()
    plt.show()
    print(f"{label}: model probability {proba_xai[i]:.3f}, LIME's local estimate {lime_result.local_pred[0]:.3f}, "
          f"local fit R^2 {lime_result.score:.2f}")

#### SHAP vs LIME: do the two explanations agree?
A check that the explanations can be trusted: when two methods that work in completely different ways tell the same story, we can rely on it more. They are **not on the same scale**: SHAP gives how much each feature's value moved this row's probability (the values add up to the prediction); LIME gives the weights of its small linear model, where each feature is turned into a range such as `grid_fixed <= 3`. So we compare the **direction** (does the feature push the probability up or down?) and the **ranking** (which features matter most?), not the exact sizes.
* **The course recipe on one row:** the podium example with the course's SHAP and LIME code.
* **The two examples:** SHAP and LIME side by side for every feature.
* **30 random validation rows:** how often the two agree on the direction of every feature with a visible SHAP effect (above 0.01), the rank correlation of their sizes (Spearman: 1 = same order, 0 = unrelated), and whether they name the same most important feature.
* **Global:** each feature's share of the total effect, averaged over the 30 rows, for both methods.

**The course recipe on one row.** First the podium example explained the way the course does it, so it is easy to check that we use SHAP and LIME correctly: a SHAP waterfall built from an explicit `shap.Explanation`, and LIME with `discretize_continuous=True` for one chosen class. Three differences from the course example, all because of our setup:
* **Class:** the course explains class 4 (`Ideal`) of a 5-class problem, so it takes `shap_values[j, :, 4]` and `labels=(4,)`. Our problem has two classes and our SHAP explainer explains only the podium probability, so SHAP needs no class index (`shap_values.values[j]`) and LIME uses `labels=(1,)` (class 1 = podium): both explain the same class.
* **Scaling:** the course gives LIME `X_train_scaled`, because its model expects scaled inputs. `FINAL_MODEL` scales inside its own pipeline, so LIME gets the raw feature values; scaling them first would scale them twice.
* **Display:** `exp.show_in_notebook()` fails with the installed IPython version (`ImportError: cannot import name 'display'`), so we display the same HTML it builds, `exp.as_html()`.

In [ ]:
from IPython.display import HTML

# The course recipe on one row: SHAP waterfall and LIME for the same row and the same class
j = examples["podium"]                         # the row to explain (the podium example of 4.11)

# SHAP, built by hand as in the course. Our explainer explains one output, the podium probability,
# so there is no class index: shap_values.values[j] instead of the course's shap_values[j, :, 4]
shap.plots.waterfall(shap.Explanation(
    values=shap_values.values[j],              # one SHAP value per feature
    base_values=shap_values.base_values[j],    # the baseline: average prediction of the background rows
    data=X_xai.iloc[j].to_numpy(),             # the feature values of row j
    feature_names=FEATURES), max_display=12, show=False)
plt.title(f"SHAP (course recipe), podium example\n{describe(j)}")
plt.tight_layout()
plt.show()

# LIME, as in the course, for the class SHAP explains: class 1 = podium
explainer_lime = LimeTabularExplainer(
    X_xai.to_numpy(),                          # feature values, NOT scaled: FINAL_MODEL scales inside its pipeline
    feature_names=FEATURES,
    class_names=["no podium", "podium"],       # class 0, class 1
    categorical_features=yes_no_columns,       # the 0/1 features
    discretize_continuous=True,                # numeric features become ranges, e.g. grid_fixed <= 5.00
    mode="classification", random_state=SEED)
exp = explainer_lime.explain_instance(
    X_xai.iloc[j].to_numpy(),                  # row j
    predict_both,                              # [P(no podium), P(podium)] for every row
    num_features=len(FEATURES),
    labels=(1,))                               # explain the podium class, the same one SHAP explains
# the course's exp.show_in_notebook(labels=(1,)) fails with the installed IPython (ImportError: display),
# so we show the same HTML it builds ourselves
display(HTML(exp.as_html(labels=(1,))))

In [ ]:
from scipy.stats import spearmanr


def lime_weights(i):
    """LIME weight of every feature (in FEATURES order) for explained row i."""
    res = lime_explainer.explain_instance(X_xai.iloc[i].to_numpy(), predict_both, num_features=len(FEATURES))
    weights = dict(res.as_map()[1])
    return np.array([weights.get(j, 0.0) for j in range(len(FEATURES))])

# 1) the two examples, side by side
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (label, i) in zip(axes, examples.items()):
    both = pd.DataFrame({"SHAP": shap_values.values[i], "LIME": lime_weights(i)}, index=FEATURES)
    both = both.loc[both["SHAP"].abs().sort_values().index]           # largest SHAP effect at the top
    both.plot.barh(ax=ax, color=["tab:blue", "tab:orange"])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"SHAP vs LIME, {label} example\n{describe(i)}", fontsize=9)
    ax.set_xlabel("SHAP value (change in probability) / LIME weight")
plt.tight_layout()
plt.show()

# 2) agreement on 30 random validation rows
sample = np.random.default_rng(SEED).choice(len(X_xai), 30, replace=False)
shap_sample = shap_values.values[sample]
lime_sample = np.array([lime_weights(i) for i in sample])
rows = []
for s, l in zip(shap_sample, lime_sample):
    visible = np.abs(s) > 0.01                                          # features with a visible SHAP effect
    rows.append({"same direction (visible SHAP effects)": (np.sign(s[visible]) == np.sign(l[visible])).mean()
                 if visible.any() else np.nan,
                 "rank correlation of sizes (Spearman)": spearmanr(np.abs(s), np.abs(l)).statistic,
                 "same most important feature": float(np.argmax(np.abs(s)) == np.argmax(np.abs(l)))})
display(pd.DataFrame(rows).mean().round(3).to_frame("average over 30 rows"))

# 3) global: each feature's share of the total effect, averaged over the 30 rows
share = pd.DataFrame({"SHAP": np.abs(shap_sample).mean(axis=0), "LIME": np.abs(lime_sample).mean(axis=0)},
                     index=FEATURES)
share = (share / share.sum()).sort_values("SHAP")
ax = share.plot.barh(figsize=(8, 5), color=["tab:blue", "tab:orange"])
ax.set_title("SHAP vs LIME: share of the total effect per feature (30 validation rows)")
ax.set_xlabel("share of the total absolute effect")
plt.tight_layout()
plt.show()
print(f"rank correlation of the global shares: {spearmanr(share['SHAP'], share['LIME']).statistic:.2f}")

**What we see (SHAP vs LIME).**
* **The course recipe (podium example):** both methods explain the same row and the same class. SHAP starts at the baseline 0.171 and its contributions add up exactly to the model's 0.910. LIME's panel shows the model's own probability (0.91) and its ranges: `grid_fixed <= 5`, `pts_before > 71`, `gap_to_pole_pct <= 0.96` and `qual_position <= 5` push towards podium, `cons_pts_before > 141` and `qg_q3=1` push away, the same directions as SHAP.
* **Podium example (Verstappen):** the two methods agree on the direction of every feature with a visible effect: points, grid slot, qualifying position and gap to pole push up, the team's points push down. They differ in size and order: SHAP puts the points first (+0.37) and the grid second (+0.31), LIME the grid first, and LIME's bars are smaller because its weights are on another scale (its local estimate was 0.33 against the model's 0.91, see above).
* **Non-podium example (Vettel):** grid 12 is the biggest push down for both methods. Qualifying position, points, gap to pole and the Q2-only flag push down in both, the Q3 flag and the team's points push up in both. Only `pos_before` disagrees, a feature with a tiny effect (SHAP -0.007).
* **30 random validation rows:** same direction for **87%** of the visible effects, rank correlation of the sizes **0.76**, same most important feature in **2 of 3** rows.
* **Global:** both methods put `grid_fixed` first (29% of the total effect for SHAP, 25% for LIME) and give the missing flags no effect (every explained row is complete); the rank correlation of the two global orders is **0.82**. LIME gives the qualifying flags (`qg_q3`, `qg_q2_only`) more weight than SHAP: it treats 0/1 features as categories and the other features as ranges, so its scale is not SHAP's.
* **Conclusion:** two methods that work in completely different ways tell the same story (the grid first, then points and qualifying, with the same directions), so the explanations in this section are consistent. Where they differ in size we trust SHAP, whose values add up exactly to the model's probability.

#### The most confident miss
The non-podium row the model was most sure about: what happened in that race?

In [ ]:
most_confident_miss = int(np.argmax(np.where(y_xai == 0, proba_xai, -1)))
print(f"most confident miss: {describe(most_confident_miss)}")

**Interpretation.**
* **The starting slot carries the model.** Per feature, shuffling `grid_fixed` drops the validation PR-AUC by 0.262, `pts_before` by 0.101, `gap_to_pole_pct` by 0.044 and `qual_position` by 0.042; the rest are below 0.03. Per group: **grid (0.261), driver standings (0.165), qualifying (0.122)**, then constructor standings (0.038). P3's ablation (retraining without each group) also puts grid, qualifying and driver standings first.
* **Per feature vs per group:** the qualifying features are twins of each other (position, gap, Q3 flag), so each one alone looks small (0.04) while the group together is 0.12: shuffling one lets the model lean on the others. **Permutation vs ablation:** permutation keeps the trained model and breaks a group; ablation retrains without it, so the retrained model leans on the remaining twins and the drops are smaller.
* **Direction (SHAP summary):** a low (blue) `grid_fixed`, `qual_position`, `gap_to_pole_pct` and championship positions push the probability up, by up to +0.3 for the grid; many points before the race (`pts_before`, red) push it up by up to +0.4. Two signs look odd and come from twin features: many *team* points (`cons_pts_before`, red) push *down*, and reaching Q3 (`qg_q3`) pushes slightly down. Given the driver's own points and qualifying position, the model uses these to correct the twins (for example, a driver with few points in a team with many points is the slower team-mate). The flags (`no_qual_time`, `standings_missing`) show no effect here because every explained row is complete.
* **Podium example (Verstappen, Abu Dhabi 2021, from pole, won):** SHAP starts at the baseline 0.171 and adds 369.5 points (+0.37), grid 1 (+0.31), qualifying P1 (+0.11) and gap to pole 0 (+0.08); his team's high points pull back (-0.20). Result: 0.910. **Non-podium example (Vettel, French GP 2021, grid 12, finished 9th):** grid 12 (-0.08), qualifying 12th (-0.03), only 28 points (-0.03), 2% off pole (-0.02) bring him down to 0.016. Both stories are what a fan would expect.
* **LIME agrees on the direction but is only an approximation:** its small linear model estimates 0.33 where the real model says 0.91 (local fit R² 0.50), and it fits the non-podium row poorly (R² 0.10). SHAP values add up exactly to the model's prediction, so we trust SHAP more for the size of each effect.
* **The most confident miss is again Hamilton at the 2020 Italian Grand Prix:** pole position, predicted 81.3%, finished 7th after a stop-and-go penalty during the race. A pre-race model cannot see events inside the race.

**Not causal.** These explanations describe the model, not Formula 1. The model relies on the grid, the championship and qualifying because they are the best pre-race summary of how fast the car and driver are; it does not show that the starting position *causes* podiums.

### 4.12 Model limitations

Four checks give numbers to the main limitations.
1. **Are the predicted probabilities realistic in every split?** Every race has exactly 3 podiums, so the predicted probabilities of one race should add up to about 3. This check needs **no labels**, so it can be run on any future season before its races; we also compare the average predicted probability with the real podium rate.
2. **Era sensitivity:** the test years are a new technical era. Is the final model equally good in each test season?
3. **The false alarms on the test set:** how many of those cars did not finish the race (crash, breakdown), something no pre-race model can see?
4. **Feature drift:** do the feature values of the test years look like those of the full-weight training years (2006-2019)? This also needs no labels.

These checks only describe the final model; nothing is changed because of them.

In [ ]:
# 1) are the probabilities realistic? (podiums per race should be about 3; this part needs no labels)
parts = [("train 1950-2005", train[train["year"] < 2006]), ("train 2006-2019", train[train["year"] >= 2006]),
         ("validation", val), ("test", test)]
calib_rows = []
for split, part in parts:
    p = pd.Series(FINAL_MODEL.predict_proba(part[FEATURES].astype(float))[:, 1], index=part.index)
    calib_rows.append({"split": split, "predicted podiums per race (median)": p.groupby(part["raceId"]).sum().median(),
                       "average predicted probability": p.mean(), "real podium rate": part["podium"].mean()})
display(pd.DataFrame(calib_rows).set_index("split").round(3))

# 2) era sensitivity: the final model on each test season separately
season_rows = []
for year, part in test.groupby("year"):
    season_rows.append({"season": year, "races": part["raceId"].nunique(),
                        **evaluate(FINAL_MODEL, part[FEATURES].astype(float), part["podium"], threshold=FINAL_THRESHOLD)})
display(pd.DataFrame(season_rows).set_index("season").round(3))

# 3) false alarms on the test set: how many of those cars did not finish the race?
status = test[["raceId", "driverId"]].merge(outcomes[["raceId", "driverId", "status"]],
                                            on=["raceId", "driverId"], how="left")["status"]
finished = (status.eq("Finished") | status.str.match(r"^\+\d+ Laps?$")).to_numpy()
said_podium = FINAL_MODEL.predict_proba(X_test)[:, 1] >= FINAL_THRESHOLD
false_alarm = said_podium & (y_test.to_numpy() == 0)
print(f"test false alarms: {false_alarm.sum()}; did not finish the race: {(~finished[false_alarm]).sum()} "
      f"({(~finished[false_alarm]).mean():.0%}), compared with {(~finished).mean():.0%} of all test starters")

# 4) feature drift (no labels): how far does each feature's test mean move, in training standard deviations?
X_modern = X_train[modern]
drift = pd.DataFrame({"train 2006-2019 mean": X_modern.mean(), "test mean": X_test.mean(),
                      "test rows above the training maximum": (X_test > X_train.max()).mean()})
drift["shift (training SDs)"] = (drift["test mean"] - drift["train 2006-2019 mean"]) / X_modern.std()
display(drift.sort_values("shift (training SDs)", key=abs, ascending=False).head(6).round(3))

**What we see.**
* **Probabilities are realistic everywhere, slightly high on test:** median predicted podiums per race 3.09 (1950-2005), 2.93 (2006-2019), 3.02 (validation) and 3.14 (test); average probability 0.159 on test against a real podium rate of 0.151. This is better than the previous version (3.32 on test).
* **The drift check explains the small excess:** the points features moved most. Before a race in 2022-2024 a driver has on average 58.8 points (36.4 in 2006-2019) and a team 118 (73): seasons now have 22-24 races and sprint races. The model learned "more points = more likely podium", so every test driver looks a little stronger. Less than 1% of test rows lie above the training maximum of any single feature, so this is a shift, not values never seen.
* **Era sensitivity, per season:** test PR-AUC 0.685 in 2022, 0.764 in 2023 and 0.635 in 2024. 2022 was the first season of new rules (P3 notes that the first season after a rule change, like 2014, was also harder); 2023 was dominated by one team and is the easiest; 2024 had four winning teams and is the hardest.
* **False alarms:** 127 on the test set; 20% of those cars did not finish the race, compared with 14% of all test starters. Some false alarms are breakdowns or crashes, which no pre-race model can see.

**Limitations of the models**

| Model | Main limitation in this project |
|---|---|
| Logistic regression (final) | Linear: no interactions unless they are built as features; the weights of twin features (points and positions, the qualifying columns, the missing flags) are not individually interpretable; it extrapolates linearly, so drifting features (points) shift its probabilities. |
| Random forest | Overfits with small leaves (train PR-AUC 0.84 vs test 0.67); probabilities are vote shares, not calibrated; cannot extrapolate beyond training values. |
| Gradient boosting | Sensitive to the number of trees (800 trees: train 0.78 vs validation 0.68) and tree size; many hyperparameters for a small validation set. |
| Shallow neural network | Stopping epoch chosen on validation, so its validation score is optimistic (0.763 on validation, 0.693 on test); wider or deeper networks did not help beyond noise; a black box without XAI. |
| k-NN, Naive Bayes (explored) | k-NN cannot use P3's training weights and weighs every feature equally; Naive Bayes assumes independent features, which our twin features break (validation PR-AUC 0.736 and 0.570). |

**Limitations of the data and the method**
* **Old seasons:** training uses 1950-2019 with the 1950-2005 rows at weight 0.25. On our validation, training on 2006-2019 only scores slightly higher (0.744 vs 0.736, within noise); the team keeps the old seasons on purpose (P3, 3.11 Part F).
* **Era sensitivity:** the test years are a new technical era (2022 ground-effect cars) that the model has never seen, and its first season is the hardest (0.685). The relative features (grid, qualifying, championship position) still transfer.
* **Points drift:** `pts_before` and `cons_pts_before` count points, and the number of points available per season has grown. P3 tested a points share (points divided by the leader's points) as a candidate and did not keep it; such a share would be more stable across eras.
* **Missing values carry meaning:** missing qualifying data is normal before 2006 but rare after it, so for a modern driver the model reads "missing" as a bad sign (4.13: a pole-sitter without qualifying data and championship position drops from 68% to 8%). For a record with a data gap the prediction is unreliable.
* **No flag for features that are never missing in training:** if such a feature (for example `grid_fixed`) is missing in a new record, it is silently replaced by the training median. These features are always known at the prediction cutoff, so a record without them should be treated as invalid input.
* **Small validation set:** 117 podiums; the bootstrap standard error of PR-AUC is about 0.04, larger than every difference between the tuned models, so the ranking of the models is uncertain.
* **The threshold is chosen for F1**, which counts a missed podium and a false alarm as equally bad. Another use weighs them differently and needs a different threshold.
* **The task itself:** incidents, penalties, safety cars, weather and strategy happen during the race and are invisible before it (Hamilton, Italy 2020, in 4.11). No pre-race model can predict them.

### 4.13 Hand-off to the inference function (P1)

P1's inference function uses the objects below. `FINAL_MODEL` is one pipeline (column selection + preprocessing + model), so it takes the columns in P3's `FEATURES` and handles missing values itself (4.2). The training weights are only used in training; a prediction needs no weight. Usage:

```python
proba = FINAL_MODEL.predict_proba(record[FEATURES])[:, 1]   # podium probability
podium = proba >= FINAL_THRESHOLD                           # yes / no decision
```

The check below runs the final model on one complete validation record and on the same record with its qualifying and championship values removed (as for a driver without a qualifying time in the first race of a season).

In [ ]:
complete_record = X_val[X_val.notna().all(axis=1)].iloc[[0]]
incomplete_record = complete_record.copy()
incomplete_record[["qual_position", "gap_to_pole_pct", "pos_before", "cons_pos_before"]] = np.nan
check = pd.concat([complete_record, incomplete_record]).assign(record=["complete", "incomplete"]).set_index("record")
check["podium probability"] = FINAL_MODEL.predict_proba(check[FEATURES])[:, 1].round(3)
check["podium (threshold)"] = check["podium probability"] >= FINAL_THRESHOLD
assert check["podium probability"].notna().all()
display(check[["grid_fixed", "qual_position", "pts_before", "pos_before", "podium probability", "podium (threshold)"]])
print(f"FINAL_NAME = {FINAL_NAME!r}, FINAL_THRESHOLD = {FINAL_THRESHOLD}, {len(FEATURES)} features")